# Rubin Cluster Injection: Presentation Figures

Produce a real Rubin background zoom, intrinsic King/Plummer/Sersic/EFF light profiles, the local Rubin PSF, convolved models, injected science images, and injected-minus-original differences. One gallery cell compares the available smooth-profile parameters. Every figure is displayed and exported as 300-dpi PNG plus PDF.

Run in the **CANFAR Rubin container/kernel** that worked for your injection test. This is a separate notebook; it does not modify the existing notebooks or remote data. The backend evaluates `getPsf().computeImage(...)` and `computeShape(...)` at the injection position. **Gaussian fallbacks are rejected**, caching is disabled, and extra source noise is disabled so the difference isolates the injected model.

Run in order. Change the dataset selection and figure controls in section 2. Images are examples, not calibrated detection/completeness measurements. Cell-boundary PSF discontinuities remain after conversion.

## 1. Kernel and checkout
Open from `/arc/home/snair25/INJECT`. If Rubin imports fail, select the LSST container/kernel rather than installing Butler into a plain environment. If project dependencies are missing, install the checkout into this kernel with `%pip install -e /arc/home/snair25/INJECT`, restart, and rerun. No packages are installed automatically.

In [ ]:
from pathlib import Path
import sys

print('Python:', sys.executable)
try:
    from lsst.daf.butler import Butler
    from lsst.geom import Point2D
    import lsst.afw.image
except ImportError as exc:
    raise ImportError('Select the CANFAR Rubin Science Pipelines kernel, then restart.') from exc

search_paths = [Path.cwd(), *Path.cwd().parents, Path('/arc/home/snair25/INJECT')]
REPO_ROOT = next((path.resolve() for path in search_paths
                  if (path / 'inject' / '__init__.py').is_file()
                  and (path / 'pyproject.toml').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the INJECT checkout or update search_paths.')
sys.path.insert(0, str(REPO_ROOT))
import inject
if not getattr(inject, '__file__', None) or Path(inject.__file__).resolve() != REPO_ROOT / 'inject' / '__init__.py':
    raise ImportError('Wrong inject package cached. Restart the kernel and use this checkout.')

import json
from getpass import getpass, GetPassWarning
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, TwoSlopeNorm
from matplotlib.patches import Rectangle
from inject import notebook_output_dir
from inject.inject import make_profile_image, get_actual_psf, inject_clusters_rubin_psf, inspect_psf_mask
from inject.light_profiles import mag_to_flux

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.titlesize': 14,
    'axes.labelsize': 12, 'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('INJECT:', inject.__file__)

## 2. Dataset, physical inputs and figure controls
DP2 uses `deep_coadd`; DP1 can use the older `deepCoadd`. By default we query five real candidates in collection `dp2`, band `g`, and select the first. Set `DATA_ID_OVERRIDE` to your working full data ID, keeping `skymap`, and use an exact run in `COLLECTION` for reproducibility.

**Physical mode is on by default.** Choose `DISTANCE_MPC`, `R_HALF_PC` (the nominal physical radius parameter), `ABSOLUTE_MAGNITUDE` in the selected image band, and band-specific `EXTINCTION_MAG`. The notebook uses the project's `apparent_magnitude_from_absolute` and `physical_radius_to_pixels` helpers to change apparent magnitude and radius in pixels together. Doubling distance halves the angular size and quarters the total flux for the same cluster. The original Rubin sky and the local PSF remain fixed. `DISTANCE_SWEEP_MPC` controls the four distances in the gallery.

These conversions use nearby-source Euclidean distance modulus and small-angle geometry. They do not model cosmological angular-diameter/luminosity distances, redshift, K corrections, or dust variation between distances. Extinction is an input in magnitudes, not a fitted foreground correction.

Set `USE_PHYSICAL_PARAMETERS=False` to retain direct `BASE_MAGNITUDE`/`BASE_R_HALF` controls without assigning a physical distance. In physical mode these two defaults are ignored; explicit radius/magnitude gallery sweeps still deliberately override one observed parameter at the base distance, changing the inferred physical source. Only the distance row moves the same intrinsic source.

`ZOOM_CENTER_XY` is relative to the lower-left of the loaded image, not sky coordinates. `None` chooses the centre of the context. Choose an odd `ZOOM_SIZE`. Every model is placed at its central pixel. Move the centre if it is masked or has no valid PSF. Leave calibration overrides unset unless you know the image units.

**Radius/sampling caveat:** the existing profile implementations have different radius conventions; `R_HALF_PC` is a nominal generator scale, not a guarantee of identical measured projected half-light radii across families. The table reports finite-stamp `r50` separately. Sampling and wing truncation affect it; subpixel intrinsic radii can be unresolved, so this notebook does not certify high-precision size recovery. Age is not varied at fixed smooth-profile magnitude.

In [ ]:
DATASET = 'dp2'
COLLECTION = 'dp2'
BAND = 'g'
DATA_ID_OVERRIDE = None
CANDIDATE_INDEX = 0
USE_TOKEN_PROMPT = False

CONTEXT_SIZE = 800
ZOOM_SIZE = 201
ZOOM_CENTER_XY = None

USE_PHYSICAL_PARAMETERS = True
DISTANCE_MPC = 5.0
R_HALF_PC = 4.0
ABSOLUTE_MAGNITUDE = -7.5
EXTINCTION_MAG = 0.0
DISTANCE_SWEEP_MPC = [1.0, 2.0, 5.0, 10.0]

BASE_MAGNITUDE = 21.0
BASE_R_HALF = 4.0
KING_CONCENTRATION = 10.0
SERSIC_INDEX = 1.5
EFF_GAMMA = 3.0
ZERO_POINT_OVERRIDE = None
PIXEL_SCALE_OVERRIDE = None

if DATASET not in ('dp1', 'dp2'):
    raise ValueError('This notebook targets CANFAR dp1 or dp2, not Rubin-hosted dp02.')
if ZOOM_SIZE % 2 != 1 or ZOOM_SIZE < 101:
    raise ValueError('Choose an odd ZOOM_SIZE of at least 101 pixels.')
REPO_URL = f'https://ws-uv.canfar.net/lsst/api/butler/repo/{DATASET}/butler.yaml'
OUTPUT_DIR = notebook_output_dir(f'presentation_{DATASET}')
print('Figures will be saved to:', OUTPUT_DIR)
print('Input mode:', 'physical luminosity/radius/distance' if USE_PHYSICAL_PARAMETERS else 'apparent magnitude/pixel radius')

## 3. Connect and select a real coadd
Use the CANFAR session's CADC credentials by default. For manual entry set `USE_TOKEN_PROMPT=True` above and enter the CADC token in the hidden prompt, without `Bearer `. Nothing writes or displays the token. Do not use a Rubin token for this CANFAR URL. Restart the kernel when finished.

Only the science exposure names below are selected, never maps or pretty-picture products. Pixel access can still fail after a metadata query succeeds.

In [ ]:
if USE_TOKEN_PROMPT:
    from lsst.daf.butler.remote_butler import RemoteButlerFactory
    factory = RemoteButlerFactory.create_factory_for_url(REPO_URL)
    with warnings.catch_warnings():
        warnings.simplefilter('error', GetPassWarning)
        access_token = getpass('CADC/CANFAR token (hidden): ').strip()
    try:
        if not access_token or access_token.lower().startswith('bearer '):
            raise ValueError('Enter the token itself without a Bearer prefix.')
        butler = factory.create_butler_for_access_token(access_token)
    finally:
        del access_token
else:
    butler = Butler(REPO_URL)

known_types = {item.name: item for item in butler.registry.queryDatasetTypes('*')}
COADD_DATASET_TYPE = next((name for name in ('deep_coadd', 'deepCoadd') if name in known_types), None)
if COADD_DATASET_TYPE is None:
    raise RuntimeError('No supported science coadd registered. Inspect the live dataset types.')
if DATA_ID_OVERRIDE is None:
    candidates = list(butler.query_datasets(
        COADD_DATASET_TYPE, collections=[COLLECTION],
        where='band = :band', bind={'band': BAND}, find_first=True, limit=5,
    ))
else:
    candidates = list(butler.query_datasets(
        COADD_DATASET_TYPE, collections=[COLLECTION],
        data_id=DATA_ID_OVERRIDE, find_first=True, limit=5,
    ))
if not candidates:
    raise RuntimeError('No matching images. Change the collection, band, or data ID.')
for index, candidate in enumerate(candidates):
    print(index, candidate.run, dict(candidate.dataId.required))
if not 0 <= CANDIDATE_INDEX < len(candidates):
    raise ValueError('Choose a candidate index printed above.')
selected = candidates[CANDIDATE_INDEX]
DATA_ID = dict(selected.dataId.required)
science_butler = butler.clone(collections=[selected.run])
print('Selected:', COADD_DATASET_TYPE, DATA_ID)

## 4. Load the background and verify the actual PSF
Modern `lsst.images.cells.CellCoadd` uses `to_legacy()`. Older cell products use `stitch().asExposure()`. Neither conversion replaces the supplied PSF with a Gaussian. The PSF is evaluated at the same full-image position used for injection.

A full coadd is downloaded before the context/zoom are selected, so `CONTEXT_SIZE` does not cap network or peak memory use. This section stops on a flagged PSF location or failed PSF evaluation; move the zoom centre or choose another candidate rather than presenting a fallback result.

In [ ]:
exposure = science_butler.get(COADD_DATASET_TYPE, dataId=DATA_ID)
print('Raw product:', f'{type(exposure).__module__}.{type(exposure).__qualname__}')
if callable(getattr(exposure, 'to_legacy', None)):
    exposure = exposure.to_legacy()
if callable(getattr(exposure, 'stitch', None)):
    exposure = exposure.stitch()
if callable(getattr(exposure, 'asExposure', None)):
    exposure = exposure.asExposure()
if any(not hasattr(exposure, name) for name in ('image', 'mask', 'getPsf', 'getBBox')):
    raise TypeError(f'Unsupported converted product: {type(exposure)}')

psf = exposure.getPsf()
if psf is None:
    raise RuntimeError('No Rubin PSF attached to this image.')
bbox = exposure.getBBox()
full_x0, full_y0 = bbox.getMinX(), bbox.getMinY()
context = exposure.image.array[:CONTEXT_SIZE, :CONTEXT_SIZE].astype(float).copy()
context_mask = exposure.mask.array[:CONTEXT_SIZE, :CONTEXT_SIZE].copy()
mask_planes = dict(exposure.mask.getMaskPlaneDict())
height, width = context.shape
if ZOOM_SIZE > min(height, width):
    raise ValueError('ZOOM_SIZE exceeds the available context dimensions.')
center_x, center_y = (width // 2, height // 2) if ZOOM_CENTER_XY is None else map(int, ZOOM_CENTER_XY)
half_zoom = ZOOM_SIZE // 2
zoom_x0, zoom_y0 = center_x - half_zoom, center_y - half_zoom
if zoom_x0 < 0 or zoom_y0 < 0 or zoom_x0 + ZOOM_SIZE > width or zoom_y0 + ZOOM_SIZE > height:
    raise ValueError('Zoom extends beyond the context. Move the centre or increase CONTEXT_SIZE.')
background = context[zoom_y0:zoom_y0 + ZOOM_SIZE, zoom_x0:zoom_x0 + ZOOM_SIZE].copy()
background_mask = context_mask[zoom_y0:zoom_y0 + ZOOM_SIZE, zoom_x0:zoom_x0 + ZOOM_SIZE].copy()
if not np.isfinite(background).all():
    raise RuntimeError('Zoom includes non-finite pixels; choose another centre.')
parent_x0, parent_y0 = full_x0 + zoom_x0, full_y0 + zoom_y0
parent_point = Point2D(float(full_x0 + center_x), float(full_y0 + center_y))
mask_status = inspect_psf_mask(background_mask, mask_planes, half_zoom, half_zoom)
if mask_status['flagged']:
    raise RuntimeError(f'PSF mask flags at the centre: {mask_status["reasons"]}. Choose another centre.')

if PIXEL_SCALE_OVERRIDE is None:
    if exposure.getWcs() is None:
        raise RuntimeError('No WCS: set a verified PIXEL_SCALE_OVERRIDE in arcsec/pixel.')
    PIXEL_SCALE = float(exposure.getWcs().getPixelScale(parent_point).asArcseconds())
else:
    PIXEL_SCALE = float(PIXEL_SCALE_OVERRIDE)
if ZERO_POINT_OVERRIDE is None:
    photo_calib = exposure.getPhotoCalib()
    if photo_calib is None:
        raise RuntimeError('No photometric calibration: set a verified ZERO_POINT_OVERRIDE.')
    zero_flux = float(photo_calib.getInstFluxAtZeroMagnitude())
    if not np.isfinite(zero_flux) or zero_flux <= 0:
        raise RuntimeError('Invalid photometric zero flux; supply a verified zero point.')
    ZERO_POINT = float(2.5 * np.log10(zero_flux))
else:
    ZERO_POINT = float(ZERO_POINT_OVERRIDE)
if not np.isfinite(PIXEL_SCALE) or PIXEL_SCALE <= 0 or not np.isfinite(ZERO_POINT):
    raise ValueError('Invalid pixel scale or zero point.')

actual_psf_gs, psf_fwhm_px = get_actual_psf(
    psf, half_zoom, half_zoom, parent_x0, parent_y0, pixel_scale=PIXEL_SCALE
)
psf_stamp = psf.computeImage(parent_point).array.astype(float).copy()
if not np.isfinite(psf_stamp).all() or psf_stamp.sum() <= 0:
    raise RuntimeError('The actual Rubin PSF image is not finite and positive in total flux.')
psf_stamp /= psf_stamp.sum()
print('PSF:', type(psf).__name__, '| FWHM:', round(psf_fwhm_px, 3), 'pixels')
print('Pixel scale:', PIXEL_SCALE, 'arcsec/pixel | AB zero point:', ZERO_POINT)
print('Injection position in parent pixels:', parent_point)
print('Rubin computeImage and computeShape succeeded; no Gaussian fallback.')

## 5. Figure and model helpers
These helpers use INJECT's `make_profile_image` and `inject_clusters_rubin_psf`, rather than a separate hand-written profile or convolution model. Flux is conserved over the rendered finite stamp. The `concentration` catalog field is overloaded by the backend: King uses a tidal/core radius ratio, Sersic uses index `n`, and EFF uses slope `gamma`. The wrapper below gives these distinct names.

The measured `r50` encloses half the flux of the finite sampled image, not an infinite analytic model. Intrinsic/model panels use flux per pixel; data panels retain the science image's native units. Individual gallery rows share a colour scale so changing magnitude really changes the displayed brightness.

For display only, logarithmic images floor non-positive/sub-threshold pixels at the colour-scale minimum. Radial curves omit values below one millionth of their own peak to avoid plotting numerical convolution noise. The numerical arrays, flux checks and radius measurements are not altered by these display choices.

In [ ]:
from inject import apparent_magnitude_from_absolute, distance_modulus, physical_radius_to_pixels

saved_figures = []
parameter_records = []

def observed_parameters(distance_mpc):
    distance_pc = float(distance_mpc) * 1e6
    radius_px = physical_radius_to_pixels(R_HALF_PC, distance_pc=distance_pc, pixel_scale=PIXEL_SCALE)
    if not np.isfinite(ABSOLUTE_MAGNITUDE) or not np.isfinite(EXTINCTION_MAG) or EXTINCTION_MAG < 0:
        raise ValueError('Absolute magnitude must be finite and extinction must be finite and nonnegative.')
    apparent_mag = apparent_magnitude_from_absolute(
        ABSOLUTE_MAGNITUDE, distance_pc=distance_pc, extinction=EXTINCTION_MAG
    )
    return apparent_mag, radius_px

if USE_PHYSICAL_PARAMETERS:
    default_apparent_mag, default_radius_px = observed_parameters(DISTANCE_MPC)
    print(f'Distance {DISTANCE_MPC:g} Mpc -> apparent magnitude {default_apparent_mag:.3f}, radius input {default_radius_px:.3f} px')

def save_figure(fig, name):
    for extension in ('png', 'pdf'):
        path = OUTPUT_DIR / f'{name}.{extension}'
        fig.savefig(path, dpi=300, bbox_inches='tight')
        saved_figures.append(str(path))
    plt.show()
    plt.close(fig)

def centred_canvas(stamp):
    if max(stamp.shape) > ZOOM_SIZE:
        raise ValueError('Model exceeds zoom size. Increase ZOOM_SIZE or reduce the radius.')
    canvas = np.zeros_like(background, dtype=float)
    start_y = half_zoom - stamp.shape[0] // 2
    start_x = half_zoom - stamp.shape[1] // 2
    canvas[start_y:start_y + stamp.shape[0], start_x:start_x + stamp.shape[1]] = stamp
    return canvas

def measured_r50(stamp):
    grid_y, grid_x = np.indices(stamp.shape)
    radius = np.hypot(grid_x - (stamp.shape[1] - 1) / 2, grid_y - (stamp.shape[0] - 1) / 2)
    weights = np.clip(stamp.astype(float), 0, None)
    order = np.argsort(radius.ravel())
    cumulative = np.cumsum(weights.ravel()[order])
    if cumulative[-1] <= 0:
        raise ValueError('Cannot measure a half-light radius for a zero-flux model.')
    return float(radius.ravel()[order][np.searchsorted(cumulative, cumulative[-1] / 2)])

def radial_curve(stamp):
    grid_y, grid_x = np.indices(stamp.shape)
    annulus = np.floor(np.hypot(grid_x - (stamp.shape[1] - 1) / 2,
                                grid_y - (stamp.shape[0] - 1) / 2)).astype(int)
    total = np.bincount(annulus.ravel(), weights=stamp.ravel())
    count = np.bincount(annulus.ravel())
    radii = (np.arange(len(total)) + 0.5) * PIXEL_SCALE
    return radii, total / np.maximum(count, 1)

def render_case(profile='king', r_half=None, magnitude=None,
                king_c=KING_CONCENTRATION, sersic_n=SERSIC_INDEX, eff_gamma=EFF_GAMMA,
                distance_mpc=None):
    if USE_PHYSICAL_PARAMETERS:
        active_distance = DISTANCE_MPC if distance_mpc is None else float(distance_mpc)
        default_mag, default_radius = observed_parameters(active_distance)
    else:
        if distance_mpc is not None:
            raise ValueError('Enable USE_PHYSICAL_PARAMETERS to specify a physical distance.')
        active_distance = None
        default_mag, default_radius = BASE_MAGNITUDE, BASE_R_HALF
    magnitude = default_mag if magnitude is None else float(magnitude)
    r_half = default_radius if r_half is None else float(r_half)
    if not np.isfinite(magnitude) or not np.isfinite(r_half) or r_half <= 0:
        raise ValueError('Magnitude must be finite and radius must be finite and positive.')
    shape_parameter = {'king': king_c, 'plummer': 10.0, 'sersic': sersic_n, 'eff': eff_gamma}[profile]
    entry = dict(id=1, x=half_zoom, y=half_zoom, profile_type=profile,
                 r_half=float(r_half), magnitude=float(magnitude), age_gyr=1.0,
                 concentration=float(shape_parameter))
    intrinsic, stamp_size = make_profile_image(entry, pixel_scale=PIXEL_SCALE, zero_point=ZERO_POINT)
    if stamp_size > ZOOM_SIZE:
        raise ValueError('Increase ZOOM_SIZE before rendering this radius.')
    flux = mag_to_flux(magnitude, ZERO_POINT)
    intrinsic = intrinsic * flux
    injected, info, _, _ = inject_clusters_rubin_psf(
        image=background, catalog=[entry], psf_obj=psf,
        bbox_x_min=parent_x0, bbox_y_min=parent_y0,
        mask_array=background_mask, mask_plane_dict=mask_planes,
        pixel_scale=PIXEL_SCALE, zero_point=ZERO_POINT, add_noise=False,
        use_actual_psf=True, use_psf_cache=False, skip_bad_psf_regions=True,
        record_psf_mask_flags=True, rng_seed=42, verbose=False, band=DATA_ID['band'],
    )
    if len(info) != 1 or info[0].get('psf_used') != 'rubin':
        modes = [row.get('psf_used') for row in info]
        raise RuntimeError(f'Real Rubin PSF required; injection returned PSF modes {modes}.')
    convolved = info[0]['stamp'].astype(float)
    if not np.isfinite(convolved).all():
        raise RuntimeError('Non-finite convolved model.')
    difference = injected.astype(float) - background
    np.testing.assert_allclose(difference, centred_canvas(convolved), rtol=1e-8, atol=1e-8)
    np.testing.assert_allclose(convolved.sum(), flux, rtol=1e-6)
    physical_radius = None if active_distance is None else R_HALF_PC * r_half / default_radius
    absolute_mag = None if active_distance is None else magnitude - distance_modulus(active_distance * 1e6) - EXTINCTION_MAG
    record = dict(profile=profile, magnitude=float(magnitude), r_half_input_px=float(r_half),
                  distance_mpc=active_distance, nominal_r_half_pc=physical_radius,
                  absolute_magnitude=absolute_mag,
                  extinction_mag=EXTINCTION_MAG if active_distance is not None else None,
                  measured_intrinsic_r50_px=measured_r50(intrinsic),
                  measured_convolved_r50_px=measured_r50(convolved),
                  king_c=float(king_c) if profile == 'king' else None,
                  sersic_n=float(sersic_n) if profile == 'sersic' else None,
                  eff_gamma=float(eff_gamma) if profile == 'eff' else None,
                  psf_used=info[0]['psf_used'], psf_fwhm_px=float(info[0]['psf_fwhm_px']))
    parameter_records.append(record)
    return dict(intrinsic=intrinsic, convolved=convolved, injected=injected,
                difference=difference, record=record)

def image_extent(stamp):
    return tuple(value * PIXEL_SCALE for value in (
        -stamp.shape[1] / 2, stamp.shape[1] / 2, -stamp.shape[0] / 2, stamp.shape[0] / 2
    ))

def show_stamp(axis, stamp, title, norm, cmap='magma'):
    display = np.maximum(stamp, norm.vmin) if isinstance(norm, LogNorm) else stamp
    artist = axis.imshow(display, origin='lower', cmap=cmap, norm=norm,
                         extent=image_extent(stamp), interpolation='nearest')
    axis.set_title(title)
    axis.set_xlabel('Offset [arcsec]')
    axis.set_ylabel('Offset [arcsec]')
    return artist

science_limits = np.percentile(background, [1, 99.5])
if science_limits[0] == science_limits[1]:
    raise RuntimeError('Constant background: choose a different zoom centre.')
science_norm = Normalize(*science_limits)

## 6. Image 1: the original square Rubin background
This cell produces one standalone square picture of the selected DP2 sky region, with no synthetic source, marker, axes, title, or PSF panel. Real stars, galaxies, sky signal and noise remain in the image; "background" means the unmodified science crop, not a fitted sky-only model.

The identical crop is used for every injection below. Background and injected images use the same grayscale stretch so their appearance can be compared directly. Files are saved as `01_dp2_background.png` and `.pdf`; PNGs are 1500 by 1500 pixels at 300 dpi.

In [ ]:
def save_square_image(image, name, norm, cmap='gray'):
    if image.shape != background.shape or image.shape[0] != image.shape[1]:
        raise ValueError('Every presentation image must match the same square sky crop.')
    display = np.maximum(image, norm.vmin) if isinstance(norm, LogNorm) else image
    fig = plt.figure(figsize=(5, 5))
    axis = fig.add_axes([0, 0, 1, 1])
    axis.imshow(display, origin='lower', cmap=cmap, norm=norm,
                extent=image_extent(image), interpolation='nearest', aspect='equal')
    axis.set_axis_off()
    for extension in ('png', 'pdf'):
        path = OUTPUT_DIR / f'{name}.{extension}'
        fig.savefig(path, dpi=300, bbox_inches=None, pad_inches=0)
        saved_figures.append(str(path))
    plt.show()
    plt.close(fig)

save_square_image(background, '01_dp2_background', science_norm)
print('Image 1: the original Rubin sky crop, without any injected source.')

## 7. Prepare the intrinsic profiles and PSF-convolved injections
Prepare King, Plummer, Sersic and EFF examples at the same position, integrated magnitude and generator radius input. This cell only prepares the arrays and reports their parameters. The next cell produces the standalone pictures.

Image 2 is the intrinsic 2D cluster light profile, before PSF convolution and without a Rubin background. It is padded to the same square field of view as image 1. The profiles share a logarithmic model colour scale. A common radius input need not imply identical measured `r50`; inspect the table. All accepted injections must use the actual Rubin PSF, without extra source noise.

In [ ]:
profile_names = ('king', 'plummer', 'sersic', 'eff')
profile_labels = {'king': 'King', 'plummer': 'Plummer', 'sersic': 'Sersic', 'eff': 'EFF'}
cases = {name: render_case(profile=name) for name in profile_names}
model_peak = max(case['intrinsic'].max() for case in cases.values())
model_norm = LogNorm(vmin=max(float(model_peak) * 1e-5, 1e-12), vmax=float(model_peak))
print('Models ready. The next cell displays images 2, 3 and 4 separately for each family.')
pd.DataFrame([case['record'] for case in cases.values()])

## 8. Images 2, 3 and 4: profile, injected sky and difference
For each profile family this cell displays and saves three separate square pictures, not a multi-panel grid. Use the shared image 1 above to complete the sequence:

1. **Original:** the unchanged square Rubin sky crop from section 6.
2. **Intrinsic profile:** the generated 2D King/Plummer/Sersic/EFF model alone, before PSF convolution (`02_<profile>_intrinsic`).
3. **Injected sky:** that same original Rubin image plus the source after convolution with its actual local Rubin PSF (`03_<profile>_background_plus_injection`). There is no separate PSF-only or convolved-model-only panel.
4. **Difference:** image 3 minus image 1 (`04_<profile>_difference`). This isolates the PSF-convolved added light; no extra source noise is generated.

Every file has the same square footprint, centre and angular extent. Titles, axes, borders and colour bars are omitted from these slide-ready files. Original and injected sky share a grayscale stretch; differences share a symmetric red/blue scale around zero. The optional parameter gallery in section 9 remains separate from this four-image sequence.

In [ ]:
difference_peak = max(float(np.max(np.abs(case['difference']))) for case in cases.values())
if difference_peak <= 0:
    raise RuntimeError('No injected signal to display.')
difference_norm = TwoSlopeNorm(vmin=-difference_peak, vcenter=0, vmax=difference_peak)

for name in profile_names:
    case = cases[name]
    np.testing.assert_allclose(case['difference'], case['injected'] - background)
    print(f'{profile_labels[name]} | image 2: intrinsic 2D light profile, before PSF convolution')
    save_square_image(centred_canvas(case['intrinsic']), f'02_{name}_intrinsic', model_norm, 'magma')
    print(f'{profile_labels[name]} | image 3: the same Rubin background plus the PSF-convolved source')
    save_square_image(case['injected'], f'03_{name}_background_plus_injection', science_norm)
    print(f'{profile_labels[name]} | image 4: injected image minus the original background')
    save_square_image(case['difference'], f'04_{name}_difference', difference_norm, 'RdBu_r')

print('Use the shared image 1 with any one profile family to form the four-image sequence.')

## 9. One-cell parameter gallery
Run the next cell to produce all parameter-comparison plots: convolved-model images, radial-profile comparisons, and intrinsic-versus-convolved enclosed-light curves. Edit the values in `sweeps` and `DISTANCE_SWEEP_MPC`. Physical mode adds a seventh row: the same intrinsic King cluster at four distances, with both its angular size and total flux recomputed. The background, PSF and source position are held fixed.

Other rows vary one observed parameter or shape parameter at the base distance. For example, the magnitude row changes intrinsic luminosity at fixed distance; it is not itself a distance experiment. The table records the effective absolute magnitude and nominal physical radius for each case.

The gallery covers family, radius, magnitude, King concentration, Sersic index, EFF slope, and (in physical mode) distance. It does not invent changes with age, ellipticity or metallicity that this smooth-profile builder does not implement. Each image row shares its own colour scale; do not compare colours between rows. Subpixel-radius results are illustrative, not precision physical half-light-radius measurements.

In [ ]:
sweeps = [
    ('Profile family', 'profile', ['king', 'plummer', 'sersic', 'eff'], 'king'),
    ('Radius input [px]', 'r_half', [2.0, 4.0, 6.0, 8.0], 'king'),
    ('AB magnitude', 'magnitude', [19.5, 21.0, 22.5, 24.0], 'king'),
    ('King concentration rt/rc', 'king_c', [3.0, 8.0, 15.0, 30.0], 'king'),
    ('Sersic index n', 'sersic_n', [0.5, 1.0, 2.0, 4.0], 'sersic'),
    ('EFF slope gamma', 'eff_gamma', [2.2, 2.5, 3.0, 4.0], 'eff'),
]
if USE_PHYSICAL_PARAMETERS:
    sweeps.append(('Distance [Mpc]', 'distance_mpc', DISTANCE_SWEEP_MPC, 'king'))
if any(len(values) != 4 for _, _, values, _ in sweeps):
    raise ValueError('This layout expects four values in every sweep, including DISTANCE_SWEEP_MPC.')
gallery = []
for title, parameter, values, family in sweeps:
    row_cases = []
    for value in values:
        options = {'profile': family, parameter: value}
        row_cases.append(render_case(**options))
    gallery.append((title, parameter, values, row_cases))

fig, axes = plt.subplots(len(gallery), 4, figsize=(15, 3.3 * len(gallery)), constrained_layout=True)
for row_index, (title, parameter, values, row_cases) in enumerate(gallery):
    row_peak = max(float(case['convolved'].max()) for case in row_cases)
    row_norm = LogNorm(vmin=max(row_peak * 1e-5, 1e-12), vmax=row_peak)
    for column_index, (value, case) in enumerate(zip(values, row_cases)):
        label = profile_labels[value] if parameter == 'profile' else f'{value:g}'
        artist = show_stamp(axes[row_index, column_index], centred_canvas(case['convolved']),
                            f'{title}: {label}', row_norm)
        axes[row_index, column_index].set_xlim(-12, 12)
        axes[row_index, column_index].set_ylim(-12, 12)
    fig.colorbar(artist, ax=list(axes[row_index]), shrink=0.7, label='Flux / pixel')
save_figure(fig, '05_parameter_gallery_convolved')

colours = ['#176b87', '#d45055', '#418546', '#9c6a19']
curve_rows = (len(gallery) + 2) // 3
fig, axes = plt.subplots(curve_rows, 3, figsize=(16, 4.5 * curve_rows), constrained_layout=True)
for axis, (title, parameter, values, row_cases) in zip(axes.ravel(), gallery):
    for colour, value, case in zip(colours, values, row_cases):
        radii, brightness = radial_curve(case['convolved'])
        positive = brightness > float(brightness.max()) * 1e-6
        label = profile_labels[value] if parameter == 'profile' else f'{value:g}'
        axis.semilogy(radii[positive], brightness[positive], color=colour, label=label, linewidth=2)
    axis.set_title(title)
    axis.set_xlabel('Radius [arcsec]')
    axis.set_ylabel('Mean model flux / pixel')
    axis.set_xlim(0, 10)
    axis.legend(frameon=False, fontsize=10)
    axis.grid(alpha=0.15)
for axis in axes.ravel()[len(gallery):]:
    axis.set_visible(False)
save_figure(fig, '06_parameter_radial_profiles')

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for colour, name in zip(colours, profile_names):
    case = cases[name]
    for axis, key in zip(axes, ('intrinsic', 'convolved')):
        stamp = case[key]
        grid_y, grid_x = np.indices(stamp.shape)
        radius = np.hypot(grid_x - (stamp.shape[1] - 1) / 2, grid_y - (stamp.shape[0] - 1) / 2)
        order = np.argsort(radius.ravel())
        cumulative = np.cumsum(np.clip(stamp.ravel()[order], 0, None))
        cumulative /= cumulative[-1]
        axis.plot(radius.ravel()[order] * PIXEL_SCALE, cumulative, color=colour,
                  label=f'{profile_labels[name]} (r50={measured_r50(stamp) * PIXEL_SCALE:.2f} arcsec)', linewidth=2)
for axis, title in zip(axes, ('Intrinsic enclosed light', 'After Rubin PSF convolution')):
    axis.axhline(0.5, color='0.5', linestyle='--', linewidth=1)
    axis.set_title(title)
    axis.set_xlabel('Radius [arcsec]')
    axis.set_ylabel('Fraction of finite-stamp flux')
    axis.set_ylim(0, 1.02)
    axis.set_xlim(0, 8)
    axis.legend(frameon=False, fontsize=9)
    axis.grid(alpha=0.15)
save_figure(fig, '07_enclosed_light_and_half_light_radius')
print('Parameter gallery complete; every accepted model used the actual Rubin PSF.')

## 10. Export the numerical examples and provenance
The table records distance (when physical mode is active), effective absolute/apparent magnitudes, nominal physical radius, input pixel radius, measured finite-stamp half-light radii, and PSF mode. The array archive contains the unchanged background and each main example's intrinsic stamp, convolved stamp, injected image and difference. Provenance includes the physical-mode settings and distance-model assumptions. No credentials are recorded. Rerunning overwrites matching figure names; change the output directory to keep multiple runs.

In [ ]:
parameter_table = pd.DataFrame(parameter_records).drop_duplicates()
if not parameter_table['psf_used'].eq('rubin').all():
    raise RuntimeError('Presentation export refused: a non-Rubin PSF result is present.')
parameter_table.to_csv(OUTPUT_DIR / 'model_parameters_and_measured_radii.csv', index=False)
arrays = {'background': background, 'psf': psf_stamp}
for name, case in cases.items():
    for key in ('intrinsic', 'convolved', 'injected', 'difference'):
        arrays[f'{name}_{key}'] = case[key]
np.savez_compressed(OUTPUT_DIR / 'presentation_image_arrays.npz', **arrays)
provenance = dict(
    dataset=DATASET, repo=REPO_URL, dataset_type=COADD_DATASET_TYPE,
    collection=selected.run, dataset_id=str(selected.id), data_id=DATA_ID,
    context_size=CONTEXT_SIZE, zoom_size=ZOOM_SIZE, context_center_xy=[center_x, center_y],
    parent_injection_xy=[full_x0 + center_x, full_y0 + center_y],
    pixel_scale_arcsec=PIXEL_SCALE, ab_zero_point=ZERO_POINT,
    physical_mode=USE_PHYSICAL_PARAMETERS,
    distance_model='nearby Euclidean small-angle; no cosmology or K correction' if USE_PHYSICAL_PARAMETERS else None,
    base_distance_mpc=DISTANCE_MPC if USE_PHYSICAL_PARAMETERS else None,
    base_nominal_r_half_pc=R_HALF_PC if USE_PHYSICAL_PARAMETERS else None,
    base_absolute_magnitude=ABSOLUTE_MAGNITUDE if USE_PHYSICAL_PARAMETERS else None,
    extinction_mag=EXTINCTION_MAG if USE_PHYSICAL_PARAMETERS else None,
    psf_class=f'{type(psf).__module__}.{type(psf).__qualname__}', psf_fwhm_px=float(psf_fwhm_px),
    add_noise=False, psf_cache=False, require_actual_psf=True,
    models=parameter_records, figures=sorted(set(saved_figures)),
)
with (OUTPUT_DIR / 'presentation_provenance.json').open('w') as handle:
    json.dump(provenance, handle, indent=2, default=str, allow_nan=False)
print('Presentation files:', OUTPUT_DIR)
for name in sorted(set(saved_figures)):
    print(Path(name).name)
parameter_table